# Chapter 2 Source Code Notebook

This notebook builds the source code for Chapter 2, **Agents That Do Not Break**.

The code builds a controlled agent scaffold with configuration, session state, memory, routing, reasoning, and evaluation pieces for SupportOps AI.

Before running the notebook, install any required packages and set the required API keys in your environment where model powered examples are used.


## Step 1: System Configuration

### `config.py`

The scaffold begins with one place for the system settings. These values make model choices, runtime limits, and cost assumptions visible before any agent behavior is added.

In [ ]:
# config.py
from dataclasses import dataclass, field
from enum import Enum

class ModelTier(Enum):
    FAST   = "claude-haiku-4-5-20251001"   # classification, routing
    SMART  = "claude-sonnet-4-6"            # drafting, complex reasoning

@dataclass
class ReasoningConfig:
    max_steps: int = 8              # hard cap on loop iterations
    step_timeout_seconds: float = 30.0
    retry_on_schema_failure: bool = True
    max_retries: int = 2

@dataclass
class CostConfig:
    cost_per_million_input: float = 3.00
    cost_per_million_output: float = 15.00
    cost_alert_threshold_usd: float = 0.50  # alert if single req exceeds

@dataclass
class SystemConfig:
    reasoning: ReasoningConfig = field(default_factory=ReasoningConfig)
    cost: CostConfig = field(default_factory=CostConfig)
    fast_model: str = ModelTier.FAST.value
    smart_model: str = ModelTier.SMART.value
    evaluation_sample_rate: float = 1.0  # evaluate 100% of requests
    log_full_model_io: bool = True

# Singleton for use across modules
DEFAULT_CONFIG = SystemConfig()


## Step 2: Session State

### `session.py`

With configuration in place, the next step is to give each interaction an explicit state object. This keeps the ticket status, actions, outputs, and errors available to the rest of the system.

In [ ]:
# session.py
import uuid
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum
from typing import Any, Optional

class SessionStatus(Enum):
    OPEN        = "open"
    ROUTING     = "routing"
    PROCESSING  = "processing"
    DRAFTING    = "drafting"
    ESCALATED   = "escalated"
    RESOLVED    = "resolved"
    FAILED      = "failed"

@dataclass
class ActionRecord:
    step_number: int
    action_type: str
    input_summary: str
    output_summary: str
    success: bool
    cost_usd: float = 0.0
    latency_ms: float = 0.0
    timestamp: datetime = field(default_factory=datetime.utcnow)

@dataclass
class Session:
    ticket_id: str
    raw_input: str
    session_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    created_at: datetime = field(default_factory=datetime.utcnow)
    status: SessionStatus = SessionStatus.OPEN
    routing_category: Optional[str] = None
    classification: Optional[Any] = None
    retrieved_context: list = field(default_factory=list)
    plan: Optional[list] = None
    actions_taken: list[ActionRecord] = field(default_factory=list)
    draft_response: Optional[str] = None
    escalated: bool = False
    escalation_reason: Optional[str] = None
    eval_result: Optional[Any] = None
    total_cost_usd: float = 0.0
    total_latency_ms: float = 0.0

    def record_action(self, action_type, input_summary,
                      output_summary, success, cost=0.0, latency=0.0):
        record = ActionRecord(
            step_number=len(self.actions_taken) + 1,
            action_type=action_type,
            input_summary=input_summary,
            output_summary=output_summary,
            success=success,
            cost_usd=cost,
            latency_ms=latency
        )
        self.actions_taken.append(record)
        self.total_cost_usd += cost
        self.total_latency_ms += latency
        return record


## Step 3: Memory Interface

### `memory/base.py`

Once the session can hold current progress, the system needs a consistent way to read and write memory. The base interface defines that contract before choosing any specific storage implementation.

In [ ]:
# memory/base.py
from abc import ABC, abstractmethod
from typing import Any, Optional

class MemoryInterface(ABC):
    """Abstract interface for all memory implementations.

    Implementations may include: InMemoryStore, RedisStore,
    VectorStore, SQLiteStore. Callers only use this interface.
    """

    @abstractmethod
    def store(self, key: str, value: Any, ttl_seconds: Optional[int] = None) -> None:
        """Store a value under a key. Optional TTL for expiry."""

    @abstractmethod
    def retrieve(self, key: str) -> Optional[Any]:
        """Return the value for key, or None if not found."""

    @abstractmethod
    def search(self, query: str, top_k: int = 5) -> list[Any]:
        """Return top_k items most relevant to query."""

    @abstractmethod
    def forget(self, key: str) -> bool:
        """Delete a key. Returns True if it existed."""

    @abstractmethod
    def list_keys(self, prefix: Optional[str] = None) -> list[str]:
        """Return all keys, optionally filtered by prefix."""


### `memory/in_memory.py`

The first concrete memory store keeps records in memory so the architecture can be tested without external infrastructure. It is simple on purpose, which makes the contract easy to understand.

In [ ]:
# memory/in_memory.py
import time
from typing import Any, Optional
from .base import MemoryInterface

class InMemoryStore(MemoryInterface):
    def __init__(self):
        self._store: dict[str, Any] = {}
        self._expiry: dict[str, float] = {}

    def store(self, key, value, ttl_seconds=None):
        self._store[key] = value
        if ttl_seconds:
            self._expiry[key] = time.time() + ttl_seconds

    def retrieve(self, key):
        if key in self._expiry and time.time() > self._expiry[key]:
            del self._store[key]
            del self._expiry[key]
            return None
        return self._store.get(key)

    def search(self, query, top_k=5):
        # Placeholder: simple substring match
        # Replace with vector similarity in Chapter 3
        results = [
            v for k, v in self._store.items()
            if isinstance(v, str) and query.lower() in v.lower()
        ]
        return results[:top_k]

    def forget(self, key):
        existed = key in self._store
        self._store.pop(key, None)
        self._expiry.pop(key, None)
        return existed

    def list_keys(self, prefix=None):
        keys = list(self._store.keys())
        if prefix:
            keys = [k for k in keys if k.startswith(prefix)]
        return keys


## Step 4: Request Router

### `router.py`

After state and memory are available, the request router decides which processing path a ticket should take. This keeps routing logic explicit instead of hiding it inside a model prompt.

In [ ]:
# router.py
import json
from dataclasses import dataclass
from enum import Enum
from typing import Optional
import anthropic
from .config import SystemConfig, DEFAULT_CONFIG
from .tracker import tracked_call, SystemTrace

class RoutingCategory(Enum):
    BILLING       = "billing"
    TECHNICAL     = "technical"
    ACCOUNT       = "account"
    ESCALATION    = "escalation"   # needs immediate human
    OTHER         = "other"

@dataclass
class RoutingDecision:
    category: RoutingCategory
    confidence: str        # high / medium / low
    requires_immediate_escalation: bool
    routing_reason: str

ROUTING_PROMPT = """Classify this customer support message.
Return JSON with:
- category: one of [billing, technical, account, escalation, other]
- confidence: one of [high, medium, low]
- requires_immediate_escalation: true if message contains legal threats,
  safety concerns, or regulatory language
- routing_reason: one sentence explaining the decision
Return ONLY valid JSON."""

def route_request(
    client: anthropic.Anthropic,
    raw_input: str,
    trace: SystemTrace,
    config: SystemConfig = DEFAULT_CONFIG
) -> RoutingDecision:
    response = tracked_call(
        client, "router", trace,
        model=config.fast_model,
        max_tokens=256,
        messages=[{"role": "user",
                   "content": f"{ROUTING_PROMPT}\n\nMessage:\n{raw_input}"}]
    )
    data = json.loads(response.content[0].text.strip())
    return RoutingDecision(
        category=RoutingCategory(data["category"]),
        confidence=data["confidence"],
        requires_immediate_escalation=data["requires_immediate_escalation"],
        routing_reason=data["routing_reason"]
    )


## Step 5: The Reasoning Loop

### `reasoning/loop.py`

The reasoning loop then controls how work progresses step by step. It tracks progress, enforces limits, records actions, and prevents the agent from running without clear boundaries.

In [ ]:
# reasoning/loop.py
import time
from dataclasses import dataclass
from enum import Enum
from typing import Callable, Optional
import anthropic
from ..session import Session, SessionStatus
from ..config import SystemConfig, DEFAULT_CONFIG
from ..tracker import SystemTrace

class StepResult(Enum):
    CONTINUE  = "continue"   # proceed to next step
    COMPLETE  = "complete"   # task is done
    ESCALATE  = "escalate"   # needs human intervention
    FAILED    = "failed"     # unrecoverable error

@dataclass
class LoopStep:
    name: str
    execute: Callable[[Session, SystemTrace], StepResult]
    required: bool = True      # if False, failure is non-fatal
    timeout_seconds: float = 30.0

class ReasoningLoop:
    def __init__(self, config: SystemConfig = DEFAULT_CONFIG):
        self.config = config
        self.steps: list[LoopStep] = []

    def add_step(self, step: LoopStep) -> 'ReasoningLoop':
        self.steps.append(step)
        return self  # fluent interface

    def run(self, session: Session, trace: SystemTrace) -> Session:
        session.status = SessionStatus.PROCESSING
        max_steps = self.config.reasoning.max_steps

        for i, step in enumerate(self.steps):
            if i >= max_steps:
                # Hard cap - never exceed configured maximum
                session.status = SessionStatus.FAILED
                session.record_action(
                    "step_cap_exceeded",
                    f"Reached max_steps={max_steps}",
                    "Loop terminated by control layer",
                    success=False
                )
                break

            start = time.time()
            try:
                result = step.execute(session, trace)
                latency = (time.time() - start) * 1000

                session.record_action(
                    step.name,
                    f"Step {i+1} of {len(self.steps)}",
                    result.value,
                    success=(result != StepResult.FAILED),
                    latency=latency
                )

                if result == StepResult.COMPLETE:
                    session.status = SessionStatus.RESOLVED
                    break
                elif result == StepResult.ESCALATE:
                    session.status = SessionStatus.ESCALATED
                    session.escalated = True
                    break
                elif result == StepResult.FAILED and step.required:
                    session.status = SessionStatus.FAILED
                    break
                # StepResult.CONTINUE: proceed to next step

            except Exception as e:
                session.record_action(
                    step.name, f"Exception in step {i+1}",
                    str(e), success=False
                )
                if step.required:
                    session.status = SessionStatus.FAILED
                    break

        return session


## Step 6: Evaluation Harness Stub

### `evaluation/harness.py`

With the loop in place, the evaluation harness gives you a starting point for checking behavior. It makes the system easier to inspect as more components are added in later chapters.

In [ ]:
# evaluation/harness.py
import time
from dataclasses import dataclass, field
from typing import Any, Callable, Optional
from ..session import Session

@dataclass
class TestCase:
    case_id: str
    description: str
    raw_input: str
    expected_routing: Optional[str] = None
    expected_classification: Optional[str] = None
    expected_escalation: Optional[bool] = None
    tags: list[str] = field(default_factory=list)

@dataclass
class EvalResult:
    case_id: str
    passed: bool
    actual_routing: Optional[str]
    actual_classification: Optional[str]
    actual_escalation: Optional[bool]
    cost_usd: float
    latency_ms: float
    notes: str = ""

class EvaluationHarness:
    def __init__(self, system_fn: Callable[[str], Session]):
        """
        system_fn: callable that takes raw_input -> Session.
        This decouples the harness from any specific system version.
        """
        self.system_fn = system_fn
        self.test_cases: list[TestCase] = []

    def add_case(self, case: TestCase) -> 'EvaluationHarness':
        self.test_cases.append(case)
        return self

    def run(self) -> list[EvalResult]:
        results = []
        for case in self.test_cases:
            start = time.time()
            session = self.system_fn(case.raw_input)
            latency = (time.time() - start) * 1000

            actual_routing = (
                session.routing_category
            )
            actual_clf = (
                session.classification.category.value
                if session.classification else None
            )

            passed = True
            if case.expected_routing and actual_routing != case.expected_routing:
                passed = False
            if case.expected_escalation is not None:
                if session.escalated != case.expected_escalation:
                    passed = False

            results.append(EvalResult(
                case_id=case.case_id,
                passed=passed,
                actual_routing=actual_routing,
                actual_classification=actual_clf,
                actual_escalation=session.escalated,
                cost_usd=session.total_cost_usd,
                latency_ms=latency
            ))
        return results

    def report(self, results: list[EvalResult]) -> dict:
        total = len(results)
        passed = sum(1 for r in results if r.passed)
        return {
            "total": total,
            "passed": passed,
            "pass_rate": passed / total if total else 0,
            "avg_cost_usd": sum(r.cost_usd for r in results) / total if total else 0,
            "avg_latency_ms": sum(r.latency_ms for r in results) / total if total else 0,
            "failures": [r.case_id for r in results if not r.passed]
        }


## Step 7: Wiring It Together

### `main.py`

The final step connects the pieces into a runnable scaffold. Configuration, session state, memory, routing, reasoning, and evaluation now work as one foundation for SupportOps AI.

In [ ]:
# main.py
import os
import anthropic
from supportops_ai.config import DEFAULT_CONFIG
from supportops_ai.session import Session, SessionStatus
from supportops_ai.router import route_request, RoutingCategory
from supportops_ai.memory.in_memory import InMemoryStore
from supportops_ai.reasoning.loop import ReasoningLoop, LoopStep, StepResult
from supportops_ai.services.classifier import classify_ticket
from supportops_ai.services.escalation import apply_escalation_rules
from supportops_ai.services.drafter import draft_response
from supportops_ai.tracker import SystemTrace
from supportops_ai.evaluation.harness import (
    EvaluationHarness, TestCase
)

client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
memory = InMemoryStore()
config = DEFAULT_CONFIG

def build_loop(session: Session, trace: SystemTrace) -> ReasoningLoop:
    """Build the reasoning loop for a given session."""

    def step_classify(sess, tr):
        clf = classify_ticket(client, sess.raw_input, tr, config)
        sess.classification = clf
        return StepResult.CONTINUE

    def step_apply_rules(sess, tr):
        sess = apply_escalation_rules(sess)
        if sess.escalated:
            return StepResult.ESCALATE
        return StepResult.CONTINUE

    def step_draft(sess, tr):
        sess.draft_response = draft_response(client, sess, tr, config)
        sess.status = SessionStatus.RESOLVED
        return StepResult.COMPLETE

    return (
        ReasoningLoop(config)
        .add_step(LoopStep("classify",       step_classify))
        .add_step(LoopStep("apply_rules",    step_apply_rules))
        .add_step(LoopStep("draft_response", step_draft))
    )

def process_ticket(ticket_id: str, raw_input: str) -> tuple[Session, SystemTrace]:
    trace = SystemTrace(ticket_id=ticket_id)
    session = Session(ticket_id=ticket_id, raw_input=raw_input)

    # Route
    routing = route_request(client, raw_input, trace, config)
    session.routing_category = routing.category.value

    if routing.requires_immediate_escalation:
        session.escalated = True
        session.escalation_reason = "immediate_escalation_flagged_by_router"
        session.status = SessionStatus.ESCALATED
        return session, trace

    # Run reasoning loop
    loop = build_loop(session, trace)
    session = loop.run(session, trace)
    return session, trace

# ── Run a test interaction ──────────────────────────────────────────
if __name__ == "__main__":
    ticket = (
        "I've been charged twice this month and my account is now suspended. "
        "I need this fixed today or I'm contacting my bank."
    )

    session, trace = process_ticket("TKT-002", ticket)

    print("=== SESSION ===")
    print(f"Status:    {session.status.value}")
    print(f"Routing:   {session.routing_category}")
    print(f"Escalated: {session.escalated}")
    if session.draft_response:
        print(f"\nDraft response:\n{session.draft_response}")
    print("\n=== ACTIONS TAKEN ===")
    for action in session.actions_taken:
        print(f"  Step {action.step_number}: {action.action_type} ")
        print(f"  -> {action.output_summary} ({action.latency_ms:.0f}ms)")
    print(f"\nTotal cost: ${session.total_cost_usd:.4f}")
    print(f"Total latency: {session.total_latency_ms:.0f}ms")

    # ── Run evaluation suite ────────────────────────────────────────
    def system_fn(raw_input: str) -> Session:
        sess, _ = process_ticket("eval", raw_input)
        return sess

    harness = (
        EvaluationHarness(system_fn)
        .add_case(TestCase(
            case_id="tc-001",
            description="Standard billing dispute",
            raw_input="I was charged twice last month.",
            expected_routing="billing",
            expected_escalation=False
        ))
        .add_case(TestCase(
            case_id="tc-002",
            description="Legal threat requires escalation",
            raw_input="I'm going to sue your company if this isn't fixed.",
            expected_escalation=True
        ))
        .add_case(TestCase(
            case_id="tc-003",
            description="Technical support request",
            raw_input="My integration stopped working after the API update.",
            expected_routing="technical",
            expected_escalation=False
        ))
    )

    results = harness.run()
    report = harness.report(results)

    print("\n=== EVALUATION REPORT ===")
    print(f"Pass rate:    {report['pass_rate']*100:.1f}%")
    print(f"Avg cost:     ${report['avg_cost_usd']:.4f}")
    print(f"Avg latency:  {report['avg_latency_ms']:.0f}ms")
    if report['failures']:
        print(f"Failed cases: {report['failures']}")
